# 1- Problema: Detectar inadimplência de cartão de credito

## 1-1 Explicação da base de dados

### Traduzido para Português

Fonte: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
Informações sobre o Conjunto de Dados

Esta pesquisa utilizou uma variável binária, inadimplência (Sim = 1, Não = 0), como variável resposta. Este estudo revisou a literatura e utilizou as seguintes 23 variáveis ​​como variáveis ​​explicativas:

LIMIT_BAL: Valor do crédito concedido (em NT$): inclui tanto o crédito individual ao consumidor quanto o crédito familiar (suplementar).

SEX: Sexo (1 = masculino; 2 = feminino).

EDUCATION: Escolaridade (1 = pós-graduação; 2 = ensino superior; 3 = ensino médio; 4 = outros).

MARRIAGE: Estado civil (1 = casado(a); 2 = solteiro(a); 3 = outros).

AGE: Idade (em anos).

PAY_1 a PAY_6: Histórico de pagamentos. Acompanhamos os registros de pagamentos mensais anteriores (de abril a setembro de 2005) da seguinte forma: PAY_1 = situação do pagamento em setembro de 2005; PAY_2 = situação do pagamento em agosto de 2005; ... PAY_6 = situação de pagamento em abril de 2005. A escala de medição da situação de pagamento é: -1 = pago em dia; 1 = atraso de um mês; 2 = atraso de dois meses; ...; 8 = atraso de oito meses; 9 = atraso de nove meses ou mais.

BILL_AMT1-BILL_AMT6: Valor da fatura (dólares taiwaneses). BILL_AMT1 = valor da fatura em setembro de 2005; BILL_AMT2 = valor da fatura em agosto de 2005; ...; BILL_AMT6 = valor da fatura em abril de 2005.

PAY_AMT1-PAY_AMT6: Valor do pagamento anterior (dólares taiwaneses). PAY_AMT1 = valor pago em setembro de 2005; PAY_AMT2 = valor pago em agosto de 2005; ...; PAY_AMT6 = valor pago em abril de 2005.


### Objetivo

Criar um modelo de machine learning com a melhor acurácia possível, que responde se o cliente irá pagar em dia a fatura do mês analisado (target)

# 2- Exploração dos Dados

## 2-1 Imports

In [1]:
import os
import pandas as pd

## Importar dados de base publica do google drive

file_id = '1iZFmr0ys8g_3BGc3jlE8UQ8hNFlOmY5s'

url = f'https://drive.google.com/uc?export=download&id={file_id}'


df = pd.read_csv(url)
display(df.head())

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


##2-2 Exploração dos features

### 2-2-1 - Ponto importante - Distribuição do target

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_1                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

In [3]:
df.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


Data set sem nulos e totalmente transformado em números

In [4]:
## Apresentação da distribuição do target: default payment next month:
target_distribution = df['default payment next month'].value_counts()
display(target_distribution)
target_percentage = df['default payment next month'].value_counts(normalize=True) * 100
print('\nPercentage Distribution:')
display(target_percentage)

,count
default payment next month,
0,23364
1,6636



Percentage Distribution:


,proportion
default payment next month,
0,77.88
1,22.12


0= Adimplente e 1 = Inadimplente
Sendo a base desbalanceada conforme distribuição acima

### 2-2-2 - Conferindo features categóricas conforme documentação

In [5]:
# Tirando o id do df
df = df.drop('ID', axis=1)

#### Feature: SEX

In [6]:
## Conferindo se as variáveis categoricas estão condizentes com documentação
## Conferiando se a feature SEX só possuí 1 e 2 e conferindo distribuição
print(df['SEX'].unique())
SEX_distribution = df['SEX'].value_counts()
display(SEX_distribution)
SEX_percentage = df['SEX'].value_counts(normalize=True) * 100
print('\nPercentage Distribution:')
display(SEX_percentage)
#SEX: Sexo (1 = masculino; 2 = feminino).

[2 1]


,count
SEX,
2,18112
1,11888



Percentage Distribution:


,proportion
SEX,
2,60.373333
1,39.626667


#### Feature: EDUCATION

In [7]:
#Conferindo se EDUCATION está condizente com:
#EDUCATION: Escolaridade (1 = pós-graduação; 2 = ensino superior; 3 = ensino médio; 4 = outros)
#cria o unique do education e imprime de forma crescente
education_unique = pd.DataFrame(sorted(df['EDUCATION'].unique()), columns=['EDUCATION'])
display(education_unique)

,EDUCATION
0,0
1,1
2,2
3,3
4,4
5,5
6,6


In [8]:
# Transformar no df os valores 0, 5 E 6 PARA 4 da coluna EDUCATION, uma vez que esses valores não estão na documentação, se transformarão em "outros"
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
#

#### Feature: MARRIAGE

In [9]:
#Confere valores da feature MARRIAGE
#MARRIAGE: Estado civil (1 = casado(a); 2 = solteiro(a); 3 = outros).
print(df['MARRIAGE'].unique())

[1 2 3 0]


In [10]:
# Transforma valor 0 para 3 (outros) da feature MARRIAGE
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

#### Feature: AGE

In [11]:
#print AGE unique ordenada
AGE_unique = pd.DataFrame(sorted(df['AGE'].unique()), columns=['AGE'])
display(AGE_unique)

,AGE
0,21
1,22
2,23
3,24
4,25
5,26
6,27
7,28
8,29
9,30


### 2-2-3 Relações de Features com Targe

In [12]:
# Correlação com a variável target ('default payment next month')
correlation_with_target = df.corr()['default payment next month'].sort_values(ascending=False)
display(correlation_with_target)

,default payment next month
default payment next month,1.000000
PAY_1,0.324794
PAY_2,0.263551
PAY_3,0.235253
PAY_4,0.216614
PAY_5,0.204149
PAY_6,0.186866
EDUCATION,0.033842
AGE,0.013890
BILL_AMT6,-0.005372


##3 - Pré-Processamento dos dados